# Quantization in Large Language Models (LLMs)

## 1. What is Quantization?

Quantization is the process of reducing the numerical precision used to represent model weights (and sometimes activations).

It reduces memory consumption and can improve inference efficiency, allowing large models to run on hardware with limited resources.

**Example:**

| Data Type | Bits per Parameter | Memory for 1B Parameters |
|---|---:|---:|
| FP32 | 32 bits | 4 GB |
| FP16 | 16 bits | 2 GB |
| INT8 | 8 bits | 1 GB |
| INT4 | 4 bits | 0.5 GB |

*These are theoretical weight-storage estimates using decimal GB, excluding runtime overhead.*

## 2. Why Do We Need Quantization?

- Reduces GPU memory requirements.
- Makes larger models accessible on consumer hardware.
- Can reduce memory bandwidth requirements.
- Can improve inference throughput and latency on supported hardware.
- Makes LLM deployment more cost-efficient.

## 3. How Does Quantization Work?

Quantization maps high-precision numerical values to a smaller set of representable values.

For example:

Original weights:
`[0.12, 0.45, -0.32, 0.89, -0.75]`

Quantized representation:
`[1, 5, -3, 7, -6]`

The quantized values are approximations of the original values.

### Affine Quantization

A simplified quantization equation is:

\[
q = \operatorname{round}\left(\frac{x}{s}\right)+z
\]

Where:
- \(x\): Original floating-point value
- \(s\): Scale factor
- \(z\): Zero-point
- \(q\): Quantized value

Dequantization approximately reconstructs the original value:

\[
\hat{x}=s(q-z)
\]

Since the reconstructed value may differ from the original, quantization introduces **quantization error**.

## 4. Types of Quantization

| Type | Description |
|---|---|
| FP16 | 16-bit floating-point representation |
| BF16 | 16-bit floating-point with a wider exponent range |
| INT8 | 8-bit integer quantization |
| INT4 | 4-bit quantization |
| FP4 | 4-bit floating-point representation |
| NF4 | 4-bit NormalFloat representation, designed for normally distributed weights |

### Two Important Training Approaches

**Post-Training Quantization (PTQ):**
- Quantize an already-trained model.
- Does not require full model retraining.
- Commonly used to reduce inference memory.

**Quantization-Aware Training (QAT):**
- Training accounts for quantization effects.
- Helps the model adapt to lower numerical precision.
- Can help preserve model quality after quantization.

## 5. Weight Quantization vs Activation Quantization

**Weight Quantization:** Reduces the precision of model parameters.

**Activation Quantization:** Reduces the precision of intermediate values generated during forward computation.

Some methods quantize only weights, while others quantize both weights and activations.

## 6. Important Libraries

| Library | Purpose |
|---|---|
| BitsAndBytes | 8-bit and 4-bit model quantization |
| Hugging Face Transformers | Loading and running quantized LLMs |
| GPTQ | Post-training weight quantization |
| AWQ | Activation-aware weight quantization |
| Quanto | Model quantization toolkit |

## 7. BitsAndBytes: Important Concepts

**BitsAndBytes** integrates low-bit quantization with Hugging Face Transformers.

Important features:

- 8-bit model loading.
- 4-bit model loading.
- FP4 and NF4 quantization.
- Double quantization.
- Higher-precision computation with quantized weights.

### NF4 (NormalFloat 4)

A 4-bit data format designed for weights that approximately follow a normal distribution.

It is particularly useful in memory-efficient fine-tuning workflows.

### Double Quantization

Quantizes the quantization constants themselves, reducing additional memory overhead.

### Compute Dtype

Specifies the precision used for computation with quantized weights, such as FP16 or BF16.

## 8. Important Trade-offs

| Advantage | Trade-off |
|---|---|
| Lower memory usage | Possible accuracy degradation |
| More accessible deployment | Hardware-dependent performance |
| Reduced memory bandwidth | Quantization overhead |
| Smaller model storage | Some methods require calibration |

**Important:** Lower precision does not automatically guarantee faster inference. Performance depends on hardware, kernels, memory bandwidth and implementation.

## 9. Quantization vs LoRA vs QLoRA

| Technique | Main Purpose |
|---|---|
| Quantization | Reduce numerical precision and memory usage |
| LoRA | Fine-tune a model using small trainable adapter matrices |
| QLoRA | Fine-tune using a quantized frozen base model and trainable LoRA adapters |

### QLoRA in One Line

**QLoRA = 4-bit quantized base model + LoRA adapters.**

## 10. Key Takeaways

1. Quantization reduces the number of bits used to represent model parameters.
2. Lower precision reduces weight-storage requirements.
3. INT8 and INT4 are common low-bit representations.
4. NF4 is particularly relevant to QLoRA.
5. Quantization introduces approximation error and may affect model quality.
6. BitsAndBytes allows us to load quantized models without implementing quantization manually.
7. Quantization is an important technique for deploying LLMs on memory-constrained hardware.

**Practical Goal:** Load the same pretrained LLM in FP16, INT8 and INT4, then compare memory consumption and generated responses.

In [1]:
# In this colab notbook we will to quantization of an LLM

In [1]:
!pip install -q -U transformers accelerate bitsandbytes

In [2]:
import torch
print("CUDA AVAILABLE:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA AVAILABLE: True
Tesla T4


In [3]:
import torch
from transformers import AutoTokenizer ,AutoModelForCausalLM
model_name="Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# load the half-prec(16 bit ) model

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("FP16 memory:", model_fp16.get_memory_footprint() / 1024**2, "MB")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

FP16 memory: 942.29296875 MB


In [4]:
#Step 3->test model generation

In [5]:
def ask_model(model, question):
    messages = [{"role": "user", "content": question}]

    # Convert messages into Qwen's chat format
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate the answer
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    # Decode only newly generated tokens
    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )


question = "Explain binary search in three simple points."

fp16_response = ask_model(model_fp16, question)

print(fp16_response)

Binary search is an efficient algorithm for finding the position of a target value within a sorted array. It works by repeatedly dividing the search interval in half. Here's a step-by-step explanation of how binary search works:

1. **Initialization**: Start with the entire range of the array.
2. **Divide and Conquer**: Divide the array into two halves until you have one element left to consider (the "half" is determined by the length of the array divided by 2).
3.


In [6]:
# NOW WE will generate answers from our 8bit model
from transformers import BitsAndBytesConfig

# Configure 8-bit quantization
int8_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# Load the same model with 8-bit weights
model_int8 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=int8_config,
    device_map="auto"
)

print("INT8 memory:", model_int8.get_memory_footprint() / 1024**2, "MB")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

INT8 memory: 601.04296875 MB


In [13]:
print(int8_response)

Binary Search is an efficient algorithm for finding the position of a target value within a sorted array. It works by repeatedly dividing the search interval in half and narrowing down the possible locations of the target until it is found or determined to be absent.

Here's a step-by-step explanation of how Binary Search works:

1. **Initialization**: Start with the entire range of the array as the initial search space.
2. **Divide and Conquer**: Divide the search space into two equal parts (or halves


In [14]:
# Configure 4-bit NF4 quantization
int4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load the same model with 4-bit weights
model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=int4_config,
    device_map="auto"
)

print("INT4 memory:", model_int4.get_memory_footprint() / 1024**2, "MB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

INT4 memory: 430.41796875 MB


In [15]:
int4_response = ask_model(model_int4, question)

print(int4_response)

Certainly! Binary search is an efficient algorithm for finding the position of a target value within a sorted array. Here's a brief explanation of its three main steps:

1. **Initialization**: 
   - The first step involves setting up the parameters:
     - `low` (the lower bound): This represents the index of the first element that could potentially contain the target.
     - `high` (the upper bound): This represents the index of the last element that could potentially contain the target.

2


In [16]:
# Let's compare alll the model

# Display memory used by each model
print("FP16:", round(model_fp16.get_memory_footprint()/1024**2, 2), "MB")
print("INT8:", round(model_int8.get_memory_footprint()/1024**2, 2), "MB")
print("INT4:", round(model_int4.get_memory_footprint()/1024**2, 2), "MB")

FP16: 942.29 MB
INT8: 601.04 MB
INT4: 430.42 MB


In [18]:
print("FP16 RESPONSE:\n\n\n", fp16_response)
print("\nINT8 RESPONSE:\n\n\n", int8_response)
print("\nINT4 RESPONSE:\n\n\n", int4_response)

FP16 RESPONSE:


 Binary search is an efficient algorithm for finding the position of a target value within a sorted array. It works by repeatedly dividing the search interval in half. Here's a step-by-step explanation of how binary search works:

1. **Initialization**: Start with the entire range of the array.
2. **Divide and Conquer**: Divide the array into two halves until you have one element left to consider (the "half" is determined by the length of the array divided by 2).
3.

INT8 RESPONSE:


 Binary Search is an efficient algorithm for finding the position of a target value within a sorted array. It works by repeatedly dividing the search interval in half and narrowing down the possible locations of the target until it is found or determined to be absent.

Here's a step-by-step explanation of how Binary Search works:

1. **Initialization**: Start with the entire range of the array as the initial search space.
2. **Divide and Conquer**: Divide the search space into two equal pa